In [1]:
import sys
from collections import defaultdict
from itertools import combinations


In [2]:
def parse_gr(path):
    n = None
    m = None
    edges = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "p":
                if len(parts) != 4:
                    raise ValueError("Invalid p-line in .gr file")

                _, problem, n_str, m_str = parts

                if problem != "tw":
                    raise ValueError("Expected problem descriptor 'tw'")

                n = int(n_str)
                m = int(m_str)

            else:
                if n is None:
                    raise ValueError("Edge appeared before p-line")

                if len(parts) != 2:
                    raise ValueError("Invalid edge line in .gr file")

                u, v = map(int, parts)

                if not (1 <= u <= n and 1 <= v <= n):
                    raise ValueError("Vertex out of range in .gr file")

                edges.append((u, v))

    if n is None or m is None:
        raise ValueError("Missing p-line in .gr file")

    if len(edges) != m:
        raise ValueError(f"Expected {m} edges, got {len(edges)}")

    return n, edges


In [3]:
def parse_td(path):
    n_bags = None
    max_bag_size = None
    n_vertices = None

    bags = {}
    tree_edges = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "s":
                if len(parts) != 5:
                    raise ValueError("Invalid s-line in .td file")

                _, problem, n_bags_str, max_bag_size_str, n_vertices_str = parts

                if problem != "td":
                    raise ValueError("Expected solution descriptor 'td'")

                n_bags = int(n_bags_str)
                max_bag_size = int(max_bag_size_str)
                n_vertices = int(n_vertices_str)

            elif parts[0] == "b":
                if n_bags is None:
                    raise ValueError("Bag appeared before s-line")

                bag_id = int(parts[1])

                if not (1 <= bag_id <= n_bags):
                    raise ValueError("Bag id out of range")

                bag_vertices = tuple(map(int, parts[2:]))

                bags[bag_id] = bag_vertices

            else:
                if n_bags is None:
                    raise ValueError("Tree edge appeared before s-line")

                if len(parts) != 2:
                    raise ValueError("Invalid tree edge line")

                a, b = map(int, parts)

                if not (1 <= a <= n_bags and 1 <= b <= n_bags):
                    raise ValueError("Tree edge contains invalid bag id")

                tree_edges.append((a, b))

    if n_bags is None:
        raise ValueError("Missing s-line in .td file")

    for i in range(1, n_bags + 1):
        if i not in bags:
            raise ValueError(f"Missing bag line for bag {i}")

    return n_bags, max_bag_size, n_vertices, bags, tree_edges


In [4]:
def build_graph(n, edges):
    adj = [set() for _ in range(n + 1)]
    looped = set()

    for u, v in edges:
        if u == v:
            looped.add(u)
        else:
            adj[u].add(v)
            adj[v].add(u)

    return adj, looped


def build_tree(n_bags, tree_edges):
    tree = [[] for _ in range(n_bags + 1)]

    for a, b in tree_edges:
        tree[a].append(b)
        tree[b].append(a)

    if n_bags > 1 and len(tree_edges) != n_bags - 1:
        raise ValueError(".td tree must have exactly N - 1 edges")

    return tree


def root_tree(tree, root):
    parent = {root: 0}
    order = [root]

    for t in order:
        for x in tree[t]:
            if x == parent[t]:
                continue
            if x in parent:
                raise ValueError(".td tree contains a cycle")
            parent[x] = t
            order.append(x)

    if len(parent) != len(tree) - 1:
        raise ValueError(".td tree is disconnected")

    children = defaultdict(list)

    for x, p in parent.items():
        if p != 0:
            children[p].append(x)

    postorder = list(reversed(order))

    return parent, children, postorder


def mask_to_set(mask, bag):
    result = set()

    for i, v in enumerate(bag):
        if mask & (1 << i):
            result.add(v)

    return result


def set_weight(vertices, weights):
    return sum(weights[v] for v in vertices)


def is_independent(vertices, adj, looped):
    for v in vertices:
        if v in looped:
            return False

    for v in vertices:
        for u in adj[v]:
            if u in vertices:
                return False

    return True


def all_independent_masks(bag, adj, looped):
    masks = []

    for mask in range(1 << len(bag)):
        chosen = mask_to_set(mask, bag)

        if is_independent(chosen, adj, looped):
            masks.append(mask)

    return masks


def compatible(parent_mask, parent_bag, child_mask, child_bag):
    parent_choice = {}

    for i, v in enumerate(parent_bag):
        parent_choice[v] = bool(parent_mask & (1 << i))

    for j, v in enumerate(child_bag):
        if v in parent_choice:
            child_has_v = bool(child_mask & (1 << j))

            if parent_choice[v] != child_has_v:
                return False

    return True


def selected_overlap_weight(parent_mask, parent_bag, child_bag, weights):
    child_vertices = set(child_bag)
    total = 0

    for i, v in enumerate(parent_bag):
        if v in child_vertices and parent_mask & (1 << i):
            total += weights[v]

    return total


def maximum_weight_independent_set(n, edges, n_bags, bags, tree_edges, weights=None):
    if weights is None:
        weights = [0] + [1] * n

    adj, looped = build_graph(n, edges)
    tree = build_tree(n_bags, tree_edges)

    root = 1
    _, children, postorder = root_tree(tree, root)

    bag_masks = {}
    dp = {}
    choice = {}

    for t in postorder:
        bag = bags[t]
        valid_masks = all_independent_masks(bag, adj, looped)
        bag_masks[t] = valid_masks

        dp[t] = {}
        choice[t] = {}

        for mask in valid_masks:
            chosen_vertices = mask_to_set(mask, bag)
            value = set_weight(chosen_vertices, weights)
            child_choices = {}

            for child in children[t]:
                best_child_value = None
                best_child_mask = None

                for child_mask in bag_masks[child]:
                    if not compatible(mask, bag, child_mask, bags[child]):
                        continue

                    overlap = selected_overlap_weight(mask, bag, bags[child], weights)
                    candidate = dp[child][child_mask] - overlap

                    if best_child_value is None or candidate > best_child_value:
                        best_child_value = candidate
                        best_child_mask = child_mask

                if best_child_value is None:
                    raise ValueError("No compatible child state found")

                value += best_child_value
                child_choices[child] = best_child_mask

            dp[t][mask] = value
            choice[t][mask] = child_choices

    best_root_mask = max(dp[root], key=lambda mask: dp[root][mask])
    best_value = dp[root][best_root_mask]

    result = set()

    def reconstruct(t, mask):
        result.update(mask_to_set(mask, bags[t]))

        for child, child_mask in choice[t][mask].items():
            reconstruct(child, child_mask)

    reconstruct(root, best_root_mask)

    return best_value, result




In [12]:
import multiprocessing as mp
import os
import sys
import time
import traceback
from collections import defaultdict
from queue import Empty


DATA_DIR = "data"
TIMEOUT_SECONDS = 60


def parse_gr(path):
    n = None
    edges = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "p":
                if len(parts) < 4:
                    raise ValueError("Invalid p-line in .gr file")

                n = int(parts[2])

            elif parts[0] == "e":
                if len(parts) != 3:
                    raise ValueError("Invalid edge line in .gr file")

                u, v = map(int, parts[1:])
                edges.append((u, v))

            else:
                if len(parts) != 2:
                    raise ValueError("Invalid edge line in .gr file")

                u, v = map(int, parts)
                edges.append((u, v))

    if n is None:
        raise ValueError("Missing p-line in .gr file")

    return n, edges


def parse_td(path):
    n_bags = None
    max_bag_size = None
    n_vertices = None

    bags = {}
    tree_edges = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "s":
                if len(parts) != 5:
                    raise ValueError("Invalid s-line in .td file")

                _, problem, n_bags_str, max_bag_size_str, n_vertices_str = parts

                if problem != "td":
                    raise ValueError("Expected solution descriptor 'td'")

                n_bags = int(n_bags_str)
                max_bag_size = int(max_bag_size_str)
                n_vertices = int(n_vertices_str)

            elif parts[0] == "b":
                if n_bags is None:
                    raise ValueError("Bag appeared before s-line")

                bag_id = int(parts[1])

                if not (1 <= bag_id <= n_bags):
                    raise ValueError("Bag id out of range")

                bags[bag_id] = tuple(map(int, parts[2:]))

            else:
                if n_bags is None:
                    raise ValueError("Tree edge appeared before s-line")

                if len(parts) != 2:
                    raise ValueError("Invalid tree edge line")

                a, b = map(int, parts)

                if not (1 <= a <= n_bags and 1 <= b <= n_bags):
                    raise ValueError("Tree edge contains invalid bag id")

                tree_edges.append((a, b))

    if n_bags is None:
        raise ValueError("Missing s-line in .td file")

    for i in range(1, n_bags + 1):
        if i not in bags:
            raise ValueError(f"Missing bag line for bag {i}")

    return n_bags, max_bag_size, n_vertices, bags, tree_edges


def build_graph(n, edges):
    adj = [set() for _ in range(n + 1)]
    looped = set()

    for u, v in edges:
        if u == v:
            looped.add(u)
        else:
            adj[u].add(v)
            adj[v].add(u)

    return adj, looped


def build_tree(n_bags, tree_edges):
    if n_bags < 1:
        raise ValueError(".td must contain at least one bag")

    if len(tree_edges) != n_bags - 1:
        raise ValueError(".td tree must have exactly N - 1 edges")

    tree = [[] for _ in range(n_bags + 1)]

    for a, b in tree_edges:
        tree[a].append(b)
        tree[b].append(a)

    return tree


def root_tree(tree, root):
    parent = {root: 0}
    order = [root]

    for t in order:
        for x in tree[t]:
            if x == parent[t]:
                continue

            if x in parent:
                raise ValueError(".td tree contains a cycle")

            parent[x] = t
            order.append(x)

    if len(parent) != len(tree) - 1:
        raise ValueError(".td tree is disconnected")

    children = defaultdict(list)

    for x, p in parent.items():
        if p != 0:
            children[p].append(x)

    return parent, children, list(reversed(order))


def mask_to_set(mask, bag):
    result = set()

    for i, v in enumerate(bag):
        if mask & (1 << i):
            result.add(v)

    return result


def mask_weight(mask, bag, weights):
    total = 0

    for i, v in enumerate(bag):
        if mask & (1 << i):
            total += weights[v]

    return total


def is_independent_mask(mask, bag, adj, looped):
    selected = []

    for i, v in enumerate(bag):
        if mask & (1 << i):
            if v in looped:
                return False

            for u in selected:
                if u in adj[v]:
                    return False

            selected.append(v)

    return True


def all_independent_masks(bag, adj, looped):
    masks = []

    for mask in range(1 << len(bag)):
        if is_independent_mask(mask, bag, adj, looped):
            masks.append(mask)

    return masks


def compatible(parent_mask, parent_bag, child_mask, child_bag):
    parent_choice = {}

    for i, v in enumerate(parent_bag):
        parent_choice[v] = bool(parent_mask & (1 << i))

    for j, v in enumerate(child_bag):
        if v in parent_choice:
            child_has_v = bool(child_mask & (1 << j))

            if parent_choice[v] != child_has_v:
                return False

    return True


def selected_overlap_weight(parent_mask, parent_bag, child_bag, weights):
    child_vertices = set(child_bag)
    total = 0

    for i, v in enumerate(parent_bag):
        if v in child_vertices and parent_mask & (1 << i):
            total += weights[v]

    return total


def maximum_weight_independent_set(
    n,
    edges,
    n_bags,
    bags,
    tree_edges,
    weights=None,
):
    if weights is None:
        weights = [0] + [1] * n

    adj, looped = build_graph(n, edges)
    tree = build_tree(n_bags, tree_edges)

    root = 1
    _, children, postorder = root_tree(tree, root)

    bag_masks = {}
    dp = {}
    choice = {}

    for t in postorder:
        bag = bags[t]
        valid_masks = all_independent_masks(bag, adj, looped)

        bag_masks[t] = valid_masks
        dp[t] = {}
        choice[t] = {}

        for mask in valid_masks:
            value = mask_weight(mask, bag, weights)
            child_choices = {}

            for child in children[t]:
                best_child_value = None
                best_child_mask = None
                overlap = selected_overlap_weight(mask, bag, bags[child], weights)

                for child_mask in bag_masks[child]:
                    if not compatible(mask, bag, child_mask, bags[child]):
                        continue

                    candidate = dp[child][child_mask] - overlap

                    if best_child_value is None or candidate > best_child_value:
                        best_child_value = candidate
                        best_child_mask = child_mask

                if best_child_value is None:
                    raise ValueError("No compatible child state found")

                value += best_child_value
                child_choices[child] = best_child_mask

            dp[t][mask] = value
            choice[t][mask] = child_choices

    best_root_mask = max(dp[root], key=lambda mask: dp[root][mask])
    best_value = dp[root][best_root_mask]

    result = set()

    def reconstruct(t, mask):
        result.update(mask_to_set(mask, bags[t]))

        for child, child_mask in choice[t][mask].items():
            reconstruct(child, child_mask)

    reconstruct(root, best_root_mask)

    return best_value, result


def solve_graph(gr_path, td_path):
    n, edges = parse_gr(gr_path)
    n_bags, _, td_n_vertices, bags, tree_edges = parse_td(td_path)

    if td_n_vertices != n:
        raise ValueError(".gr and .td disagree on number of graph vertices")

    value, independent_set = maximum_weight_independent_set(
        n=n,
        edges=edges,
        n_bags=n_bags,
        bags=bags,
        tree_edges=tree_edges,
    )

    return value, sorted(independent_set)


def worker(gr_path, td_path, queue):
    try:
        value, independent_set = solve_graph(gr_path, td_path)
        queue.put(("ok", value, independent_set))
    except Exception:
        queue.put(("error", traceback.format_exc()))


def run_with_timeout(gr_path, td_path, timeout_seconds):
    queue = mp.Queue()
    process = mp.Process(target=worker, args=(gr_path, td_path, queue))

    start = time.perf_counter()
    process.start()
    process.join(timeout_seconds)
    elapsed = time.perf_counter() - start

    if process.is_alive():
        process.terminate()
        process.join()
        return "timeout", None, elapsed

    try:
        result = queue.get(timeout=1)
    except Empty:
        return "error", "Process ended without returning a result", elapsed

    return result[0], result[1:], elapsed

def main(argv=None):
    if argv is None:
        argv = sys.argv[1:]

    cleaned_args = []
    skip_next = False

    for arg in argv:
        if skip_next:
            skip_next = False
            continue

        if arg == "-f":
            skip_next = True
            continue

        if arg.startswith("--f="):
            continue

        cleaned_args.append(arg)

    data_dir = DATA_DIR
    timeout_seconds = TIMEOUT_SECONDS

    if len(cleaned_args) > 2:
        print("Usage: python mwis_td.py [data_dir] [timeout_seconds]")
        sys.exit(1)

    if len(cleaned_args) >= 1:
        data_dir = cleaned_args[0]

    if len(cleaned_args) == 2:
        timeout_seconds = int(cleaned_args[1])

    if not os.path.isdir(data_dir):
        print(f"Data folder not found: {data_dir}")
        sys.exit(1)

    gr_files = sorted(f for f in os.listdir(data_dir) if f.endswith(".gr"))

    if not gr_files:
        print(f"No .gr files found in {data_dir}/")
        sys.exit(1)

    for gr_file in gr_files:
        base = os.path.splitext(gr_file)[0]
        gr_path = os.path.join(data_dir, gr_file)
        td_path = os.path.join(data_dir, base + ".td")

        print(f"\n=== {base} ===")

        if not os.path.exists(td_path):
            print("Error: missing matching .td file")
            continue

        status, payload, elapsed = run_with_timeout(
            gr_path=gr_path,
            td_path=td_path,
            timeout_seconds=timeout_seconds,
        )

        if status == "ok":
            value, independent_set = payload
            print(value)
            print(" ".join(map(str, independent_set)))
            print(f"Time: {elapsed:.3f}s")

        elif status == "timeout":
            print(f"Timed out after {timeout_seconds}s")
            print(f"Time: {elapsed:.3f}s")

        else:
            error_message = payload[0] if isinstance(payload, tuple) else payload
            print("Error:")
            print(error_message)
            print(f"Time: {elapsed:.3f}s")

            
if __name__ == "__main__":
    mp.freeze_support()
    main()


=== AhrensSzekeresGeneralizedQuadrangleGraph_3 ===
6
5 14 17 19 22 23
Time: 11.877s

=== BalancedTree_3_5 ===
273
2 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 